In [1]:
from dotenv import load_dotenv
 
load_dotenv()  # Load environment variables from .env file

True

### 1. Reading and Parsing PDF documents into a string text

In [3]:
import os
import pdfplumber
data_folder = '../data'  # Path to the folder containing PDF files
files = [file for file in os.listdir(data_folder)]

extracted_text = ''  # Initialize all_text to store text from new PDF files

for file_name in files:
    # Extract text from the PDF file
    pdf_path = os.path.join(data_folder, file_name)

    with pdfplumber.open(pdf_path) as pdf:
        for pdf_page in pdf.pages:
            single_page_text = pdf_page.extract_text()
            # Separate each page's text with newline
            extracted_text = extracted_text + '\n' + single_page_text
print(files)       
print(extracted_text)

['Retail_Internet_Banking_FAQs_new.pdf']

FAQs of Retail i-Net Banking
S. No. Frequently Asked Questions Answers
1 Login to IDBI Bank Retail i-Net A customer should have a valid Customer ID and i-Net
Banking Password (Login password) to log into i-Net Banking.
2 URL for Internet Banking https://inet.idbibank.co.in/
or
Customers may Login Retail Net Banking by selecting
Personal option from the dropdown available in the
Home Page of https://www.idbibank.in/
3 i-Net Banking Registration Customers with active Debit Card may click First
Time User? Register Now link available in i-Net
Banking Login Page
or
Visit any IDBI Bank Branch and submit Registration
form for i-Net Banking
4 How to Reset Password? Users with active Debit Card may click on Generate
Online Password/Forgot Password link available in i-
Net Banking Login Page
or
Visit any IDBI Bank Branch and submit the request
5 How to set i-Net Banking Users with active debit card may click on Generate
View/Transaction right? Online Pas

### 2. Chunking the extracted text using Langchain `RecursiveCharacterTextSplitter`

In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    separators=["\n\n", "\n", " ", ""],
    chunk_size=512,
    chunk_overlap=100,
    length_function=len,
)
chunks = splitter.split_text(extracted_text)
print(chunks)

d:\VS_Code\Projects\LangChain_CSB\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


['FAQs of Retail i-Net Banking\nS. No. Frequently Asked Questions Answers\n1 Login to IDBI Bank Retail i-Net A customer should have a valid Customer ID and i-Net\nBanking Password (Login password) to log into i-Net Banking.\n2 URL for Internet Banking https://inet.idbibank.co.in/\nor\nCustomers may Login Retail Net Banking by selecting\nPersonal option from the dropdown available in the\nHome Page of https://www.idbibank.in/\n3 i-Net Banking Registration Customers with active Debit Card may click First', '3 i-Net Banking Registration Customers with active Debit Card may click First\nTime User? Register Now link available in i-Net\nBanking Login Page\nor\nVisit any IDBI Bank Branch and submit Registration\nform for i-Net Banking\n4 How to Reset Password? Users with active Debit Card may click on Generate\nOnline Password/Forgot Password link available in i-\nNet Banking Login Page\nor\nVisit any IDBI Bank Branch and submit the request\n5 How to set i-Net Banking Users with active debit 

### 3. Get an embedding model

In [5]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

# Make sure to set your Google API key in the environment variables or .env file with the name "GOOGLE_API_KEY"
# os.environ["GOOGLE_API_KEY"] = "your-free-gemini-api-key"

embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-001")

embeddings

GoogleGenerativeAIEmbeddings(client=<google.genai.client.Client object at 0x0000026CC4AB1E10>, model='gemini-embedding-001', task_type=None, google_api_key=SecretStr('**********'), credentials=None, vertexai=None, project=None, location=None, base_url=None, additional_headers=None, client_args=None, api_version=None, request_options=None, output_dimensionality=None)

### Generate embeddings for the chunks (using embedding model) and store in A VECTOR STORES

Implementing for both the vector stores:

| Feature | FAISS | ChromaDB |
| :--- | :--- | :--- |
| **Created By** | Meta (Facebook AI Research) | Chroma Core (Independent Company) |
| **Cost** | 100% Free (Open-Source) | 100% Free (Open-Source) |
| **Primary Design** | Low-level mathematical execution library | Developer-friendly, full-scale database |
| **LangChain Package** | `langchain_community` (Core Utility) | `langchain-chroma` (Partner Package) |
| **Storage Handling** | In-Memory only (Requires manual file dump) | Flexible (In-Memory or Auto Disk Persistence) |
| **Required Installation**| `pip install faiss-cpu` | `pip install chromadb langchain-chroma` |
| **Metadata Filtering** | Complex (Requires writing custom code structures) | Native (Built-in simple dict filtering out of the box) |
| **Best Used For** | Fast prototyping, massive scale brute-force math | Long-term local storage, full-stack application RAG |

#### 3.1 Using ChromaDB

In [ ]:
# from langchain_community.vectorstores import Chroma
from langchain_chroma import Chroma


# This saves the data directly onto your disk in a folder named "chroma_db"
# "from_texts" method writes the embeddings to the Chroma vector store and persists it to disk.
# Don't run it multiple times for else it will create duplicate entries in the database.
vector_store = Chroma.from_texts(
    texts=chunks, 
    embedding=embeddings, 
    persist_directory="../chroma_db"
)



In [ ]:
# The proper, production-ready way TO ADD MORE ROWS to an existing database is 
# to use the .add_texts() method on your existing vector_store object.

# Yes, from_texts appends new rows by default if you point it to an existing persist_directory. 
# However, using from_texts over and over again is not the most efficient way to add new data because it re-initializes the entire class structure every time.

# This method keeps your database connection open 
# and simply pushes new text streams directly into your existing SQLite tables and HNSW index graph.
vector_store.add_texts(texts=["Just a plain text chunk with no manual ID."])

In [7]:
from langchain_chroma import Chroma

In [8]:
# To not write data and just read data from the existing "chroma_db" folder, you can use the following code:

db_path = "../chroma_db"
vector_store = Chroma(persist_directory=db_path, embedding_function=embeddings)
vector_store

##### 3.1.1 Different search methods

Different search methids:
- similarity_search(query, k=3)
- similarity_search_with_score(query, k=3)
- vector_store.as_retriever(search_kwargs={"k": 3})   : this returns a retreiver object not relevant chunks, internal it does first smilarity seacrh

Difference distance measurment formulas:

| Parameter Value | Math / Metric | Score Behavior | Best Used For |
| :--- | :--- | :--- | :--- |
| **`"l2"`** *(Default)* | Squared L2 (Euclidean Distance) | `0.0` is an absolute identical match. Scores can scale higher than `1.0`. | Default absolute geometric distance. |
| **`"cosine"`** | Cosine Distance | Formulated as $1 - \text{Cosine Similarity}$. Scale sits between `0.0` (identical) and `2.0`. | Text search where document length varies heavily. |
| **`"ip"`** | Inner Product / Dot Product | Formulated as $1 - (\text{Vector } A \cdot \text{Vector } B)$. | Specialized models or pre-normalized vectors. |

In [ ]:
# similarity_search

query = "What to do when no password?"

# k=3 retrieves the top 3 most relevant matches
docs = vector_store.similarity_search(query, k=3)

# Print out the results
for i, doc in enumerate(docs):
    print(f"\n--- Match #{i+1} ---")
    print(f"Content: {doc.page_content}")
    print(f"Metadata: {doc.metadata}")


--- Match #1 ---
Content: Password/Forgot Password link on Login Page and
follow the process or visit any IDBI Bank Branch.
10 If Transaction Password is disabled! Transaction password will be disabled if user enters
incorrect Transaction password for 3 consecutive
times.
11 How to Enable Transaction Password User has to regenerate new transaction password.
if disabled? Active Debit Card Holders may click on Generate
Online Password/Forgot Password link on Login Page
and follow the process or visit any IDBI Bank Branch.
1 | P age
Metadata: {}

--- Match #2 ---
Content: Limit? Transaction Limit Inquiry.
8 If Login Password is disabled? with Login password will be disabled if user enters
below error message "Your i-Net incorrect Login password for 5 consecutive times.
Banking login is disabled"
9 How to Enable Login Password if User has to regenerate new Login password. Active
disabled? Debit Card Holders may click on Generate Online
Password/Forgot Password link on Login Page and
follo

In [ ]:
# similarity_search_with_score

query = "What to do when no password?"

docs_with_scores = vector_store.similarity_search_with_score(query, k=3)

for doc, score in docs_with_scores:
    print(f"\nDistance Score: {score:.4f} (Lower = closer match)")
    print(f"Content: {doc.page_content}")


Distance Score: 0.6554 (Lower = closer match)
Content: Password/Forgot Password link on Login Page and
follow the process or visit any IDBI Bank Branch.
10 If Transaction Password is disabled! Transaction password will be disabled if user enters
incorrect Transaction password for 3 consecutive
times.
11 How to Enable Transaction Password User has to regenerate new transaction password.
if disabled? Active Debit Card Holders may click on Generate
Online Password/Forgot Password link on Login Page
and follow the process or visit any IDBI Bank Branch.
1 | P age

Distance Score: 0.6624 (Lower = closer match)
Content: Limit? Transaction Limit Inquiry.
8 If Login Password is disabled? with Login password will be disabled if user enters
below error message "Your i-Net incorrect Login password for 5 consecutive times.
Banking login is disabled"
9 How to Enable Login Password if User has to regenerate new Login password. Active
disabled? Debit Card Holders may click on Generate Online
Password

##### REwrite the vector store to use cosine similarity instead of the default Euclidean distance. This is done by specifying the "hnsw:space" parameter in the collection metadata when creating the Chroma vector store.

In [12]:
# REwrite the vector store to use cosine similarity instead of the default Euclidean distance. This is done by specifying the "hnsw:space" parameter in the collection metadata when creating the Chroma vector store.

vector_store = Chroma.from_texts(
    texts=chunks, 
    embedding=embeddings,
    persist_directory="../chroma_cosine_db",
    collection_metadata={"hnsw:space": "cosine"} # ◄ Switch to Cosine here
)

In [13]:
# 3. Query the database
# The scores returned here will be Cosine Distance (closer to 0.0 means highly similar)

query = "What to do when no password?"
results = vector_store.similarity_search_with_score(query, k=3)

for doc, score in results:
    print(f"Cosine Distance: {score:.4f} | Content: {doc.page_content[:50]}...")

Cosine Distance: 0.3277 | Content: Password/Forgot Password link on Login Page and
fo...
Cosine Distance: 0.3312 | Content: Limit? Transaction Limit Inquiry.
8 If Login Passw...
Cosine Distance: 0.3323 | Content: 3 i-Net Banking Registration Customers with active...


vieweing chormadb

In [ ]:
# pd.reset_option("^display\.")

In [19]:
import pandas as pd

# 2. Extract the underlying native Chroma client collection
collection = vector_store._collection

# 3. Fetch EVERYTHING (include documents, metadatas, and raw embedding float vectors)
raw_data = collection.get(include=["documents", "metadatas", "embeddings"])

# 4. Process the dictionary payload into a structured Pandas DataFrame
database_rows = []
total_records = len(raw_data["ids"])

for i in range(total_records):
    database_rows.append({
        "ID": raw_data["ids"][i],
        "Text Content": raw_data["documents"][i] if raw_data["documents"] else "None",
        "Metadata": raw_data["metadatas"][i] if raw_data["metadatas"] else {},
        # Capture the entire unrounded 768-dimension embedding list
        "Full Vector Array": raw_data["embeddings"][i]
    })

df_chroma = pd.DataFrame(database_rows)

# 6. Print the table view to your terminal
df_chroma[["ID", "Text Content", "Metadata", "Full Vector Array"]]

,ID,Text Content,Metadata,Full Vector Array
0,586dba72-1d36-41a2-aea9-d2eeec5a101f,FAQs of Retail i-Net Banking\nS. No. Frequentl...,None,"[0.010008350946009159, 0.020255671814084053, -..."
1,44e90771-36b0-4cc4-8cef-eddd08522a18,3 i-Net Banking Registration Customers with ac...,None,"[0.016131335869431496, 0.013354504480957985, -..."
2,1db59949-7a1b-46de-877f-38288a447987,5 How to set i-Net Banking Users with active d...,None,"[0.009058038704097271, -0.0005879222881048918,..."
3,34c325d4-8a34-4b0b-9bec-4a4b37f67d60,Limit? Transaction Limit Inquiry.\n8 If Login ...,None,"[0.018260685727000237, 0.01528877578675747, -0..."
4,1bbb9402-2cba-47cf-8d46-c0d818aee868,Password/Forgot Password link on Login Page an...,None,"[0.02102774940431118, 0.016767682507634163, -0..."
5,bcc2a376-7d12-4b95-af72-5be5b1a2cc82,and follow the process or visit any IDBI Bank ...,None,"[0.011653699912130833, 0.004678674507886171, 0..."
6,8d5c7b34-d37a-448d-b2c9-1e56b3e227ea,13 I am entering correct OTP but the While ent...,None,"[0.014068086631596088, 0.0119645856320858, -0...."
7,f93f1529-362e-4eac-84f7-43b40bfcbcb2,15 If the below error is displayed Click on Ge...,None,"[0.02057008445262909, 0.008381734602153301, -0..."
8,b92162a0-c262-4b4b-96b6-23360cf83db8,automatically in order to avoid unauthorized a...,None,"[0.0064520929008722305, 0.02427973411977291, -..."
9,c2a53109-addc-44dd-9c90-c64b1bd075f8,enters at the first time of Login to i- Settin...,None,"[0.02895307168364525, 0.015719672664999962, -0..."


In [28]:
import sqlite3
import pandas as pd

path = 'D:\VS_Code\Projects\LangChain_CSB\chroma_cosine_db\chroma.sqlite3'
# Connect directly to Chroma's underlying SQLite file
conn = sqlite3.connect(path)


# chroma_query = "SELECT * FROM sqlite_master WHERE type='table';"
chroma_query = """
    SELECT 
        e.id AS [Chunk ID],
        m.key AS [Metadata Key],
        m.string_value AS [Metadata Value]
    FROM embeddings e
    LEFT JOIN embedding_metadata m ON e.id = m.id
    LIMIT 10;
    """

# Query the database system catalog to see all tables
tables_df = pd.read_sql_query(chroma_query, conn)
# print("--- Actual SQLite Tables Inside Chroma ---")
# print(tables_df)

conn.close()

print("--- Actual SQLite Tables Inside Chroma ---")
tables_df

--- Actual SQLite Tables Inside Chroma ---


,Chunk ID,Metadata Key,Metadata Value
0,5,chroma:document,Password/Forgot Password link on Login Page an...
1,3,chroma:document,5 How to set i-Net Banking Users with active d...
2,13,chroma:document,on Login page) the bank for further informatio...
3,4,chroma:document,Limit? Transaction Limit Inquiry.\n8 If Login ...
4,11,chroma:document,22 How many attempts does user get to The no. ...
5,2,chroma:document,3 i-Net Banking Registration Customers with ac...
6,1,chroma:document,FAQs of Retail i-Net Banking\nS. No. Frequentl...
7,7,chroma:document,13 I am entering correct OTP but the While ent...
8,12,chroma:document,Contact the bank for further information.\nLog...
9,9,chroma:document,automatically in order to avoid unauthorized a...


#### 3.2 Using FAISS as vector store

In [22]:
# Step 3 Replacement:
from langchain_community.vectorstores import FAISS

vector_store = FAISS.from_texts(chunks, embeddings)
vector_store

In [34]:
vector_store.index_to_docstore_id

{0: '04752766-9983-4fa3-b4ae-e26304937d66',
 1: 'beb697ec-73c7-4ac7-bace-70a2ce93c9e1',
 2: 'ac68d05f-0cb8-45f2-aa72-c515a35c2de2',
 3: '04b2c079-580a-4152-a700-5775747c0b39',
 4: 'a90c88bc-8356-43df-b55e-fa0b56087061',
 5: '3dc7113b-64fd-4aab-b667-dff61f568f17',
 6: 'b2482d2b-25ee-418d-a72a-84ebadb71261',
 7: '8b427ca5-9537-4982-9f88-a9444ec88563',
 8: '333bf284-b16e-42c7-9406-66ef66a6fd52',
 9: 'fc96bfa0-274d-46c9-9639-a35c641e18ff',
 10: '4f8c14e0-ec69-4279-82d0-a61202667109',
 11: 'dcf63b11-23b9-494b-a8c5-d6c4754393b1',
 12: '5663cd08-de8b-4bac-8fb6-ca13a07ea355'}

In [28]:
print(vector_store)
print(vector_store.embeddings)
print(vector_store.index)
print(vector_store.docstore)
print(vector_store.embedding_function)

client=<google.genai.client.Client object at 0x000002B49EB1C550> model='gemini-embedding-001' task_type=None google_api_key=SecretStr('**********') credentials=None vertexai=None project=None location=None base_url=None additional_headers=None client_args=None api_version=None request_options=None output_dimensionality=None
<faiss.swigfaiss.IndexFlatL2; proxy of <Swig Object of type 'faiss::IndexFlatL2 *' at 0x000002B4CA4E3A70> >
client=<google.genai.client.Client object at 0x000002B49EB1C550> model='gemini-embedding-001' task_type=None google_api_key=SecretStr('**********') credentials=None vertexai=None project=None location=None base_url=None additional_headers=None client_args=None api_version=None request_options=None output_dimensionality=None


### Retriever

In [27]:
retriever = vector_store.as_retriever(search_kwargs={"k": 3})
retriever

VectorStoreRetriever(tags=['FAISS', 'GoogleGenerativeAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000002B4CA4E08D0>, search_kwargs={'k': 3})

### 4.Prompt

In [29]:
from langchain_core.prompts import PromptTemplate

# 4. Formulate the contextual prompt pattern
template = """Answer the question based only on the following context:
{context}

Question: {question}
"""
prompt = PromptTemplate.from_template(template)
prompt

PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='Answer the question based only on the following context:\n{context}\n\nQuestion: {question}\n')

### LLM

In [ ]:

from langchain_groq import ChatGroq


# Make sure to set your GROQ API key in the environment variables or .env file with the name "GROQ_API_KEY"
# os.environ["GROQ_API_KEY"] = "your-free-groq-api-key"

llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    max_tokens=200
)
llm

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.11'}}, output_version=None, profile={'name': 'Llama 3.3 70B Versatile', 'release_date': '2024-12-06', 'last_updated': '2024-12-06', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 32768, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x000002B4A2D26C90>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000002B4C30BFD10>, model_name='llama-3.3-70b-versatile', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None, max_tokens=200)

### Chain

In [32]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# 5. Chain the components using LCEL pipe notation
rag_chain = (
    {"context": retriever, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

# 6. Execute the workflow
response = rag_chain.invoke("What are the key takeaways from the text?")
print(response)

The key takeaways from the text are:

1. If a user's login is disabled, they should contact the bank for further information.
2. Login password will be disabled if a user enters an incorrect login password 5 consecutive times.
3. To enable a disabled login password, users can click on the "Generate Online Password/Forgot Password" link on the login page or visit an IDBI Bank Branch.
4. If a transaction is disabled, users can click on the "Generate Online Password/Forgot Password" link on the login page to set access rights to transaction access.
5. The session will time out after 5 minutes of inactivity to avoid unauthorized access.


Exploring data in vectordb of FAISS (in memory database)

In [43]:
import pandas as pd
import numpy as np

# 1. Extract the underlying FAISS index data structures
faiss_index = vector_store.index
total_vectors = faiss_index.ntotal
all_vectors = faiss_index.reconstruct_n(0, total_vectors)
index_to_id = vector_store.index_to_docstore_id
docstore = vector_store.docstore

# 2. Loop through the items and build a dataset list
database_rows = []
for i in range(total_vectors):
    chunk_id = index_to_id[i]
    document = docstore.search(chunk_id)
    
    database_rows.append({
        "Chunk ID": f"chunk_{i+1}",
        "LangChain Internal ID": chunk_id,
        "Text Content": document.page_content,
        "Vector Dimensions": faiss_index.d,
        "Full Vector Array": list(all_vectors[i]) # Storing the full 768 floats array
    })

# 3. Construct the Pandas DataFrame
df_faiss = pd.DataFrame(database_rows)

# 4. Print a clean summary to the VS Code Terminal
df_faiss[["Chunk ID", "Text Content", "Full Vector Array", "Vector Dimensions"]]

,Chunk ID,Text Content,Full Vector Array,Vector Dimensions
0,chunk_1,FAQs of Retail i-Net Banking\nS. No. Frequentl...,"[0.010008351, 0.020255672, -0.0067301453, -0.0...",3072
1,chunk_2,3 i-Net Banking Registration Customers with ac...,"[0.016131336, 0.0133545045, -0.00073175726, -0...",3072
2,chunk_3,5 How to set i-Net Banking Users with active d...,"[0.009058039, -0.0005879223, -0.005161229, -0....",3072
3,chunk_4,Limit? Transaction Limit Inquiry.\n8 If Login ...,"[0.018260684, 0.015288775, -0.0047441605, -0.0...",3072
4,chunk_5,Password/Forgot Password link on Login Page an...,"[0.02102775, 0.016767683, -0.0028216506, -0.05...",3072
5,chunk_6,and follow the process or visit any IDBI Bank ...,"[0.0116537, 0.0046786745, 0.0016007242, -0.047...",3072
6,chunk_7,13 I am entering correct OTP but the While ent...,"[0.014068086, 0.011964585, -0.008612333, -0.04...",3072
7,chunk_8,15 If the below error is displayed Click on Ge...,"[0.020570084, 0.008381735, -0.0052201697, -0.0...",3072
8,chunk_9,automatically in order to avoid unauthorized a...,"[0.006452093, 0.024279734, -0.010270249, -0.05...",3072
9,chunk_10,enters at the first time of Login to i- Settin...,"[0.028953072, 0.015719673, -0.005718663, -0.05...",3072
